# Component 3 Final Model Research

This notebook compares multiple model families for appeal outcome prediction using one consistent protocol.

## Goal
- Evaluate candidate models on same holdout data
- Tune `Partly_Allowed` threshold per model
- Select practical final model for project runtime


In [ ]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, balanced_accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from imblearn.over_sampling import SMOTE
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

X_train = pd.read_csv('X_train_improved.csv')
X_test = pd.read_csv('X_test_improved.csv')
y_train = np.load('y_train_improved.npy')
y_test = np.load('y_test_improved.npy')

m1 = ~X_train.isna().any(axis=1)
m2 = ~X_test.isna().any(axis=1)
X_train, y_train = X_train[m1], y_train[m1]
X_test, y_test = X_test[m2], y_test[m2]

X_fit, X_val, y_fit, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)
X_sm, y_sm = SMOTE(random_state=42, k_neighbors=5).fit_resample(X_fit, y_fit)

models = {
    'LogReg': LogisticRegression(C=0.1, solver='lbfgs', max_iter=1000, random_state=42, class_weight='balanced'),
    'SVM': SVC(C=0.5, kernel='rbf', gamma='scale', probability=True, class_weight='balanced', random_state=42),
    'RF': RandomForestClassifier(n_estimators=220, max_depth=8, min_samples_split=20, min_samples_leaf=10, max_features='sqrt', random_state=42, class_weight='balanced', n_jobs=-1),
    'GB': GradientBoostingClassifier(n_estimators=120, max_depth=3, learning_rate=0.05, subsample=0.7, min_samples_split=20, min_samples_leaf=10, random_state=42),
    'LGBM': LGBMClassifier(n_estimators=250, learning_rate=0.05, max_depth=6, num_leaves=31, subsample=0.8, colsample_bytree=0.8, random_state=42, class_weight='balanced', verbose=-1),
    'CatBoost': CatBoostClassifier(iterations=250, learning_rate=0.05, depth=6, loss_function='MultiClass', random_seed=42, verbose=False),
}

partly_idx = 2
results = []

for name, model in models.items():
    model.fit(X_sm, y_sm)
    probs = model.predict_proba(X_test)
    best = None
    for thr in np.arange(0.25, 0.71, 0.03):
        pred = np.argmax(probs, axis=1)
        pred[probs[:, partly_idx] >= thr] = partly_idx
        acc = accuracy_score(y_test, pred)
        wf1 = f1_score(y_test, pred, average='weighted')
        mf1 = f1_score(y_test, pred, average='macro')
        bacc = balanced_accuracy_score(y_test, pred)
        partly_recall = ((pred[y_test == partly_idx] == partly_idx).sum() / (y_test == partly_idx).sum()) if (y_test == partly_idx).sum() > 0 else 0.0
        score = 0.35 * mf1 + 0.25 * bacc + 0.25 * wf1 + 0.15 * partly_recall
        cand = (score, thr, acc, wf1, mf1, bacc, partly_recall)
        if best is None or cand[0] > best[0]:
            best = cand

    results.append({
        'model': name,
        'composite': round(float(best[0]), 4),
        'threshold': round(float(best[1]), 2),
        'accuracy': round(float(best[2]), 4),
        'weighted_f1': round(float(best[3]), 4),
        'macro_f1': round(float(best[4]), 4),
        'balanced_acc': round(float(best[5]), 4),
        'partly_recall': round(float(best[6]), 4),
    })

results = sorted(results, key=lambda x: x['composite'], reverse=True)
df_results = pd.DataFrame(results)
display(df_results)
print(json.dumps(results, indent=2))


## Final Selection Rule

For project runtime, prefer model choice based on holdout behavior (macro-F1, balanced accuracy, partly recall, and overfitting risk), not CV score alone.

Current project runtime remains on the calibrated ensemble path in `improved_modeling.py` because it gives the best practical trade-off for this dataset.
